In [0]:
%sql
DESCRIBE tseries.tseries.bronze_machines;

col_name,data_type,comment
machineID,int,null
model,string,null
age,int,null
_rescued_data,string,null


In [0]:
%sql
DESCRIBE tseries.tseries.bronze_telemetry;

col_name,data_type,comment
datetime,timestamp,null
machineID,int,null
volt,double,null
rotate,double,null
pressure,double,null
vibration,double,null
_rescued_data,string,null


In [0]:
%sql
DESCRIBE tseries.tseries.bronze_errors;
    



col_name,data_type,comment
datetime,timestamp,null
machineID,int,null
errorID,string,null
_rescued_data,string,null


In [0]:
%sql
DESCRIBE tseries.tseries.bronze_maint;


col_name,data_type,comment
datetime,timestamp,null
machineID,int,null
comp,string,null
_rescued_data,string,null


In [0]:
%sql
DESCRIBE tseries.tseries.bronze_failures;

col_name,data_type,comment
datetime,timestamp,null
machineID,int,null
failure,string,null
_rescued_data,string,null


1.data cleaning


In [0]:
%sql
CREATE OR REPLACE TABLE tseries.tseries.silver_machines AS
SELECT
    machineID,
    TRIM(model) AS model,
    age
FROM tseries.tseries.bronze_machines
WHERE machineID IS NOT NULL
  AND model IS NOT NULL
  AND TRIM(model) <> ''
  AND age IS NOT NULL
  AND age >= 0
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY machineID
    ORDER BY machineID
) = 1;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE tseries.tseries.silver_telemetry AS
SELECT
    datetime,
    machineID,
    volt,
    rotate,
    pressure,
    vibration
FROM tseries.tseries.bronze_telemetry
WHERE datetime IS NOT NULL
  AND machineID IS NOT NULL
  AND volt IS NOT NULL
  AND rotate IS NOT NULL
  AND pressure IS NOT NULL
  AND vibration IS NOT NULL
  AND volt >= 0
  AND rotate >= 0
  AND pressure >= 0
  AND vibration >= 0
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY datetime, machineID
    ORDER BY datetime
) = 1;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE tseries.tseries.silver_errors AS
SELECT
    datetime,
    machineID,
    TRIM(errorID) AS errorID
FROM tseries.tseries.bronze_errors
WHERE datetime IS NOT NULL
  AND machineID IS NOT NULL
  AND errorID IS NOT NULL
  AND TRIM(errorID) <> ''
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY datetime, machineID, errorID
    ORDER BY datetime
) = 1;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE tseries.tseries.silver_maint AS
SELECT
    datetime,
    machineID,
    UPPER(TRIM(comp)) AS comp
FROM tseries.tseries.bronze_maint
WHERE datetime IS NOT NULL
  AND machineID IS NOT NULL
  AND comp IS NOT NULL
  AND TRIM(comp) <> ''
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY datetime, machineID, comp
    ORDER BY datetime
) = 1;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE tseries.tseries.silver_failures AS
SELECT
    datetime,
    machineID,
    UPPER(TRIM(failure)) AS failure
FROM tseries.tseries.bronze_failures
WHERE datetime IS NOT NULL
  AND machineID IS NOT NULL
  AND failure IS NOT NULL
  AND TRIM(failure) <> ''
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY datetime, machineID, failure
    ORDER BY datetime
) = 1;

num_affected_rows,num_inserted_rows


In [0]:
%sql
show tables in tseries.tseries;

database,tableName,isTemporary
tseries,bronze_errors,false
tseries,bronze_failures,false
tseries,bronze_machines,false
tseries,bronze_maint,false
tseries,bronze_telemetry,false
tseries,silver_errors,false
tseries,silver_failures,false
tseries,silver_machines,false
tseries,silver_maint,false
tseries,silver_telemetry,false


In [0]:
%sql
SELECT * 
FROM tseries.tseries.silver_machines
LIMIT 10;

machineID,model,age
1,model3,18
2,model4,7
3,model3,8
4,model3,7
5,model3,2
6,model3,7
7,model3,20
8,model3,16
9,model4,7
10,model3,10


checking nulls

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(machineID) AS machine_ids,
    COUNT(model) AS models,
    COUNT(age) AS ages
FROM tseries.tseries.silver_machines;

total_rows,machine_ids,models,ages
100,100,100,100


In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(datetime) AS timestamps,
    COUNT(machineID) AS machine_ids,
    COUNT(volt) AS volt_values,
    COUNT(rotate) AS rotate_values,
    COUNT(pressure) AS pressure_values,
    COUNT(vibration) AS vibration_values
FROM tseries.tseries.silver_telemetry;

total_rows,timestamps,machine_ids,volt_values,rotate_values,pressure_values,vibration_values
876100,876100,876100,876100,876100,876100,876100


In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(datetime) AS timestamps,
    COUNT(machineID) AS machine_ids,
    COUNT(errorID) AS error_ids
FROM tseries.tseries.silver_errors;

total_rows,timestamps,machine_ids,error_ids
3919,3919,3919,3919


In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(datetime) AS timestamps,
    COUNT(machineID) AS machine_ids,
    COUNT(comp) AS components
FROM tseries.tseries.silver_maint;

total_rows,timestamps,machine_ids,components
3286,3286,3286,3286


In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(datetime) AS timestamps,
    COUNT(machineID) AS machine_ids,
    COUNT(failure) AS failures
FROM tseries.tseries.silver_failures;

total_rows,timestamps,machine_ids,failures
761,761,761,761
